# Lecture: Faster Sampling with DDIM

In **C4-2** we trained a DDPM and generated images by running the reverse
process for **all 1000 timesteps** — one network evaluation per step. The samples
were sharp and diverse, but sampling was **slow**: a single batch needs 1000
forward passes through the U-Net. This is *the* practical weakness of DDPMs, and
the one the "Try It Yourself" section of C4-2 asked you to think about.

**Denoising Diffusion Implicit Models** (DDIM, Song et al., 2021) solve exactly
this. The key insight:

> The trained network is just a **noise predictor** $\varepsilon_\theta(x_t, t)$.
> Nothing forces us to undo the noise in 1000 tiny Markov steps — we can take
> **larger strides** along a shorter, non-Markovian path and still arrive at a
> clean image.

Crucially, DDIM **reuses the model from C4-2 unchanged** — *no retraining*. We
only swap the sampling rule. In this notebook we will:

1. Load the pre-trained DDPM checkpoint from C4-2.
2. Sample with DDIM using 1000, 100, 50, 20, and 10 steps and compare quality.
3. See that DDIM with $\eta = 0$ is **deterministic** — the same start noise
   always maps to the same image.
4. Measure the speed-up directly.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./

### Load the pre-trained DDPM

DDIM needs **no new training**. We instantiate the same `DDPM` as in C4-2 and load
the 50-epoch Fashion-MNIST checkpoint. The variance schedule
($\beta_t, \alpha_t, \bar\alpha_t$) stored inside the model is all DDIM needs.

In [ ]:
import time
import torch
import matplotlib.pyplot as plt
from Diffusion import DDPM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

TIMESTEPS = 1000
CHANNELS  = 64

model = DDPM(timesteps=TIMESTEPS, channels=CHANNELS).to(device)
#model.load_model(path="models/ddpm_fashion_mnist_50epochs.pth", device=device)            # for running locally
model.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_50epochs.pth", device=device)  # for running in colab
model.eval()

### The DDIM update rule

Recall that at any timestep the network predicts the noise
$\varepsilon_\theta(x_t, t)$. From it we can immediately estimate the **clean
image**:

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t}\,\varepsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}$$

DDIM then jumps directly to the next (less noisy) timestep $s < t$ in the chosen
sub-sequence:

$$x_s = \sqrt{\bar\alpha_s}\,\hat{x}_0 + \sqrt{1 - \bar\alpha_s - \sigma^2}\,\varepsilon_\theta(x_t, t) + \sigma\,\epsilon$$

The parameter $\eta$ controls the noise scale $\sigma$:

- $\eta = 0$: $\sigma = 0$ — **fully deterministic** (classic DDIM). No randomness
  after the initial $x_T$.
- $\eta = 1$: recovers the **stochastic DDPM** ancestral sampler from C4-2.

This is implemented in `DDPM.ddim_sample` — open `Diffusion.py` to see the few
lines involved. Because we only visit `steps` timesteps instead of all 1000,
sampling is roughly $1000 / \text{steps}$ times faster.

### Quality vs. number of steps

We generate the **same** 8 images (same random seed → same starting noise $x_T$)
with a decreasing number of DDIM steps. With $\eta = 0$ the only thing that
changes is the step count, so the rows are directly comparable: fewer steps means
faster sampling but coarser images. The remarkable result is how few steps DDIM
needs to stay recognisable.

In [ ]:
step_counts = [1000, 100, 50, 20, 10]
n = 8

fig, axes = plt.subplots(len(step_counts), n, figsize=(14, 9))
for row, steps in enumerate(step_counts):
    torch.manual_seed(0)  # same start noise x_T for every row
    samples = model.ddim_sample(n, steps=steps, eta=0.0, device=device).cpu()
    samples = (samples + 1) / 2
    for col in range(n):
        axes[row, col].imshow(samples[col].squeeze().clamp(0, 1), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"{steps} steps", rotation=0, labelpad=40,
                            fontsize=11, va="center")
plt.suptitle(r"DDIM ($\eta=0$): same start noise, fewer steps top$\to$bottom", y=1.05)
plt.tight_layout()
plt.show()

### How much faster is it?

Let us time the sampling of a batch for each step count. The runtime scales
almost linearly with the number of steps, because the cost is dominated by the
U-Net forward passes. Compared to the full 1000-step DDPM sampler, 50-step DDIM
is about **20× faster** for a barely perceptible drop in quality.

In [ ]:
n = 16
print(f"{'steps':>8} | {'time (s)':>9} | {'speed-up vs 1000':>16}")
print("-" * 42)

timings = {}
for steps in step_counts:
    torch.manual_seed(0)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    _ = model.ddim_sample(n, steps=steps, eta=0.0, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    timings[steps] = time.time() - t0

base = timings[1000]
for steps in step_counts:
    print(f"{steps:>8} | {timings[steps]:>9.3f} | {base / timings[steps]:>15.1f}x")

### Deterministic sampling ($\eta = 0$)

A defining property of DDIM with $\eta = 0$: the mapping from the initial noise
$x_T$ to the final image is **deterministic**. The same seed always yields the
same image, regardless of how many steps we take (only the fidelity changes).

This is what makes diffusion models usable like a GAN's latent space: a fixed
$x_T$ is a reproducible "address" of an image. We verify determinism by sampling
twice with the same seed and checking the results are identical.

In [ ]:
torch.manual_seed(123)
a = model.ddim_sample(4, steps=50, eta=0.0, device=device)
torch.manual_seed(123)
b = model.ddim_sample(4, steps=50, eta=0.0, device=device)

print("Identical for same seed:", torch.allclose(a, b))

# Different seed -> different images
torch.manual_seed(999)
c = model.ddim_sample(4, steps=50, eta=0.0, device=device)
print("Different for new seed: ", not torch.allclose(a, c))

### Deterministic interpolation in noise space

Because $\eta = 0$ DDIM is a deterministic function of $x_T$, we can **interpolate
between two noise vectors** and decode each one — just like the GAN latent
interpolation in C3-1, but here in the diffusion model's noise space. The
intermediate images transition smoothly, which is only meaningful *because*
sampling is deterministic.

In [ ]:
torch.manual_seed(0)
z_a = torch.randn(1, 1, 28, 28, device=device)
z_b = torch.randn(1, 1, 28, 28, device=device)

n_steps = 8
alphas = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2.2))
for i, a in enumerate(alphas):
    # Spherical interpolation keeps the variance of the noise roughly constant.
    z = torch.sin((1 - a) * torch.pi / 2) * z_a + torch.sin(a * torch.pi / 2) * z_b

    # Inject the interpolated noise as the DDIM starting point.
    x = z.clone()
    step_indices = torch.linspace(0, TIMESTEPS - 1, 50, device=device).long().flip(0)
    with torch.no_grad():
        for j, step in enumerate(step_indices):
            t = torch.full((1,), step, device=device, dtype=torch.long)
            eps = model.model(x, t)
            ab = model.alphas_bar[step]
            ab_next = model.alphas_bar[step_indices[j + 1]] if j < len(step_indices) - 1 else torch.tensor(1.0, device=device)
            x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)
            x = torch.sqrt(ab_next) * x0 + torch.sqrt(1 - ab_next) * eps

    img = (x.squeeze().cpu() + 1) / 2
    axes[i].imshow(img.clamp(0, 1), cmap="gray")
    axes[i].axis("off")
    axes[i].set_title(f"{a:.2f}", fontsize=8)
plt.suptitle("Deterministic DDIM interpolation in noise space", y=1.05)
plt.tight_layout()
plt.show()

### DDPM vs. DDIM — Summary

| | DDPM (C4-2) | DDIM (C4-3) |
|---|---|---|
| Reverse steps | All $T$ (e.g. 1000) | A sub-sequence (e.g. 20–50) |
| Sampling speed | Slow | **~20–50× faster** |
| Stochastic? | Yes (ancestral) | $\eta=0$: deterministic; $\eta=1$: = DDPM |
| Retraining needed | — | **None** — reuses the DDPM model |
| Reproducible from $x_T$ | No | **Yes** (when $\eta=0$) |

The takeaway: **training and sampling are decoupled**. We trained one noise
predictor in C4-2, and DDIM lets us trade sampling steps for speed *after the
fact*, without touching the weights. This decoupling is what makes large
diffusion models like Stable Diffusion practical — they too are trained once and
sampled with a fast solver (DDIM and its successors).

---
## Try It Yourself — Experiment with DDIM

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Find the breaking point.** Lower the step count below 10 (try 5, then 3,
then 2). At how few steps do the images stop being recognisable clothing? What
does this tell you about the minimum number of network evaluations the model
needs?

**B. The role of $\eta$.** Re-run the quality grid with `eta=1.0` instead of
`0.0`. The samples should now look like the **stochastic** DDPM sampler from
C4-2. Are they still reproducible for a fixed seed? Why not? (Hint: where does
the extra randomness enter?)

**C. Speed vs. quality trade-off.** Using the timing table, pick the step count
you would choose for (i) a quick interactive preview and (ii) a final
high-quality render. Justify each choice in one sentence.

**D. Why deterministic interpolation works.** The interpolation cell only
produces smooth transitions because $\eta = 0$. Predict what the interpolation
would look like with $\eta = 1$, then try it. Why does stochasticity break the
smoothness?
